# markers: printable ArUco corner markers and ChArUco calibration board (M1)

Replaces the old `scripts/make_markers.py`.

* The four board-corner markers (DICT_4X4_50, ids 0..3 next to a1, h1, h8, a8, edge
  `markers.size` from `gambit_bringup/config/board.yaml`) as PNGs with a one-module white
  quiet zone, sized to print at 300 DPI. The copies in `gambit_bringup/models/aruco_markers/`
  were made the same way (346 px with quiet zone for a 22 mm marker).
* The ChArUco calibration board from `configs/camera.yaml: charuco`, as one PNG to print.

Writing files is a side effect, so it only happens in the last cell when `WRITE = True`.

Milestone: M1.

**TODO**
- [ ] print both at 100% scale, measure the printed marker / square edge with a ruler and
      update `board.yaml` (markers.size) and `camera.yaml` (charuco.square_length, marker_length)

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

%run ./homography.ipynb
%run ./calibration.ipynb

## Sizes
Pixels for a physical length at a given DPI: `round(length_m / 0.0254 * dpi)`.

In [ ]:
DPI = 300


def meters_to_pixels(length_m: float, dpi: int = DPI) -> int:
    return int(round(length_m / 0.0254 * dpi))


def marker_png(marker_id: int, size_m: float, dictionary: str = "DICT_4X4_50", dpi: int = DPI,
               quiet_modules: int = 1) -> np.ndarray:
    """Grayscale marker image with a white quiet zone of `quiet_modules` modules on each side.

    The marker itself (black border included) is size_m wide when printed at `dpi`.
    """
    import cv2
    side = meters_to_pixels(size_m, dpi)
    m = cv2.aruco.generateImageMarker(get_dictionary(dictionary), int(marker_id), side)
    bits = get_dictionary(dictionary).markerSize + 2   # data bits + black border, in modules
    q = int(round(side / bits)) * quiet_modules
    return cv2.copyMakeBorder(m, q, q, q, q, cv2.BORDER_CONSTANT, value=255)


def corner_marker_images(geom: BoardGeometry | None = None, dpi: int = DPI) -> dict[str, np.ndarray]:
    """{filename: image} for the four board-corner markers, e.g. marker_0_a1.png."""
    geom = geom or load_board_geometry()
    return {f"marker_{geom.ids[n]}_{n}.png": marker_png(geom.ids[n], geom.marker_size,
                                                        geom.dictionary, dpi)
            for n in MARKER_CORNERS}


def charuco_png(charuco_cfg=None, dpi: int = DPI, margin_m: float = 0.01) -> np.ndarray:
    """Printable ChArUco board image (white margin of margin_m around it)."""
    import cv2
    c = charuco_cfg or load_config("configs/camera.yaml").charuco
    board = make_charuco_board(c)
    w = meters_to_pixels(c.squares_x * c.square_length, dpi)
    h = meters_to_pixels(c.squares_y * c.square_length, dpi)
    m = meters_to_pixels(margin_m, dpi)
    return cv2.copyMakeBorder(board.generateImage((w, h)), m, m, m, m, cv2.BORDER_CONSTANT, value=255)


def write_all(out_dir: str | Path, dpi: int = DPI) -> list[Path]:
    """Write the 4 corner markers and charuco_board.png into out_dir."""
    import cv2
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, img in {**corner_marker_images(dpi=dpi), "charuco_board.png": charuco_png(dpi=dpi)}.items():
        cv2.imwrite(str(out_dir / name), img)
        paths.append(out_dir / name)
    return paths

## Self-check
Generate into a temporary folder and detect the markers back.

In [ ]:
import tempfile

import cv2

_mk = corner_marker_images()
assert sorted(_mk) == ["marker_0_a1.png", "marker_1_h1.png", "marker_2_h8.png", "marker_3_a8.png"]
assert _mk["marker_0_a1.png"].shape == (346, 346)   # same as gambit_bringup/models/aruco_markers
for _name, _im in _mk.items():
    _found = detect_markers(_im, "DICT_4X4_50")
    assert list(_found) == [int(_name.split("_")[1])], (_name, list(_found))
with tempfile.TemporaryDirectory() as _t:
    _paths = write_all(_t)
    assert len(_paths) == 5 and all(p.exists() for p in _paths)
    _cb = cv2.imread(str(Path(_t) / "charuco_board.png"))
    assert detect_charuco(_cb, make_charuco_board())[1] is not None
print("markers OK")

## Write the files (opt-in)
Set `WRITE = True` to regenerate. Default target is a local folder; the bringup copies in
`gambit_bringup/models/aruco_markers/` belong to that package, so only overwrite them on
purpose.

In [ ]:
WRITE = False
OUT_DIR = GV_ROOT / "markers_out"      # or REPO_ROOT / "gambit_bringup/models/aruco_markers"
if WRITE:
    print(write_all(OUT_DIR))